# Stage 6｜正式 Hybrid 筛选执行

本 Notebook 只编排现有正式 Stage 6 实现：锁定唯一的已完成 Hybrid Stage 5 来源，依次完成候选导入、表达式兼容性审计、Train 指标复用与预筛选、fresh Validation、六项 hard filter、Train long-excess 去相关，并产出 **Provisional Factor Pool**。这里的 source snapshot 不是 Factor Pool freeze；Notebook 不读取 Test/OOS，也不执行 D1。

## Cell 1｜Configuration

In [ ]:
from pathlib import Path
import gc
import json
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from factor_gfn.backtest import (
    CandidateSourceSpec,
    Stage6CandidateEvaluator,
    audit_expression_compatibility,
    build_stage6_evaluation_context,
    import_candidate_source_set,
    materialize_source_set,
    run_current_stage6_train_preparation,
    run_current_stage6_two_phase_provisional_selection,
    run_current_stage6_validation_evaluation,
    run_stage6_hybrid_train_reuse_overlay,
)
from factor_gfn.gfn import (
    RealRewardDataConfig,
    RealRewardDataPaths,
    RealRewardProvider,
    build_real_reward_data_context,
)

HYBRID_RUN_ID = 'hybrid_5_15_k16_seed42_20260816T025559Z'
STAGE5_RUN_DIR = (
    REPO_ROOT / 'runs' / 'stage5_hybrid_variance_real_5_15' / HYBRID_RUN_ID
)
STAGE6_ROOT = REPO_ROOT / 'runs' / 'stage6' / 'hybrid_provisional' / HYBRID_RUN_ID
DATA_PATHS = RealRewardDataPaths()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def compact_progress(event):
    event_type = event.get('event_type', 'progress')
    completed = event.get('completed', event.get('completed_count'))
    total = event.get('candidate_count', event.get('total'))
    if completed is None or completed % 100 == 0 or completed == total:
        suffix = '' if completed is None else f' | completed={completed}'
        if total is not None:
            suffix += f'/{total}'
        print(f'[{event_type}]{suffix}')

print('唯一 Stage 5 source:', STAGE5_RUN_DIR)
print('Stage 6 output root:', STAGE6_ROOT)

## Cell 2｜Preflight

只做轻量、只读检查。完整的 checkpoint/provenance/fingerprint 校验由下一步现有 source writer fail-closed 执行；任一检查失败都会停止。

In [ ]:
import torch

required = {
    name: STAGE5_RUN_DIR / name
    for name in (
        'hybrid_run_config.json',
        'runner_state.json',
        'checkpoint_latest.pt',
        'hybrid_diagnostics.jsonl',
        'train_candidate_artifact.json',
    )
}
missing = [str(path) for path in required.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(f'STOP: Stage 5 正式输入缺失: {missing}')

run_config = read_json(required['hybrid_run_config.json'])
runner_state = read_json(required['runner_state.json'])
candidate_artifact = read_json(required['train_candidate_artifact.json'])
last_diagnostic = json.loads(
    required['hybrid_diagnostics.jsonl'].read_text(encoding='utf-8').splitlines()[-1]
)
checkpoint = torch.load(required['checkpoint_latest.pt'], map_location='cpu', weights_only=False)

checks = {
    'run_id': STAGE5_RUN_DIR.name == HYBRID_RUN_ID,
    'hybrid architecture': run_config.get('objective_mode') == 'hybrid_variance',
    'run complete': runner_state.get('complete') is True,
    'pending assignment cleared': runner_state.get('pending_assignment') is None,
    'optimizer step = 1500': runner_state.get('global_optimizer_step') == 1500,
    'trajectories = 24000': runner_state.get('total_trajectories_seen') == 24000,
    'runner/artifact step aligned': (
        runner_state.get('global_optimizer_step')
        == candidate_artifact.get('committed_optimizer_step')
    ),
    'runner/diagnostics step aligned': (
        runner_state.get('global_optimizer_step')
        == last_diagnostic.get('global_optimizer_step')
    ),
    'runner/checkpoint step aligned': (
        runner_state.get('global_optimizer_step')
        == checkpoint.get('global_optimizer_step')
    ),
    'checkpoint schema aligned': (
        run_config.get('checkpoint_schema') == checkpoint.get('schema')
    ),
    'config fingerprint aligned': (
        run_config.get('config_fingerprint') == checkpoint.get('config_fingerprint')
    ),
}
failed = [name for name, passed in checks.items() if not passed]
for name, passed in checks.items():
    print(f'{name}:', 'PASS' if passed else 'FAIL')
del checkpoint
gc.collect()
if failed:
    raise RuntimeError(f'STOP: Stage 5 preflight failed: {failed}')
print('Preflight: PASS；下一 Cell 将执行现有 writer 的完整 Hybrid snapshot 校验。')

## Cell 3｜Stage 6 Source Snapshot

该 snapshot 只固定本轮 Stage 6 的唯一输入来源，不是 Factor Pool freeze。

In [ ]:
source_spec = CandidateSourceSpec(
    source_id=HYBRID_RUN_ID,
    source_type='hybrid_train_artifact',
    source_role='formal_discovery',
    source_path=STAGE5_RUN_DIR,
    approval_note='completed 100-cycle Raw Daily Hybrid Stage 5; approved as sole Stage 6 source',
)
SOURCE_SET_MANIFEST = materialize_source_set(
    [source_spec],
    STAGE6_ROOT / 'source_snapshots',
    mode='provisional',
)
source_set = read_json(SOURCE_SET_MANIFEST)
sources = source_set.get('sources', [])
if len(sources) != 1 or sources[0].get('source_id') != HYBRID_RUN_ID:
    raise RuntimeError('STOP: source set 不是唯一正式 Hybrid source')
if sources[0].get('source_type') != 'hybrid_train_artifact':
    raise RuntimeError('STOP: source set 混入 legacy/resource-limited source')
print('Source-set manifest:', SOURCE_SET_MANIFEST)
print('Source run ID:', sources[0]['source_id'])
print('Hybrid source count:', len(sources))

## Cell 4｜Candidate Import + Compatibility Audit

In [ ]:
CANDIDATE_IMPORT_MANIFEST = import_candidate_source_set(
    SOURCE_SET_MANIFEST, STAGE6_ROOT / 'candidate_import'
)
COMPATIBILITY_MANIFEST = audit_expression_compatibility(
    CANDIDATE_IMPORT_MANIFEST,
    SOURCE_SET_MANIFEST,
    STAGE6_ROOT / 'compatibility',
)
candidate_import = read_json(CANDIDATE_IMPORT_MANIFEST)
compatibility = read_json(COMPATIBILITY_MANIFEST)
print('Candidate-import manifest:', CANDIDATE_IMPORT_MANIFEST)
print('Candidate counts:', candidate_import.get('counts', candidate_import.get('record_counts')))
print('Compatibility manifest:', COMPATIBILITY_MANIFEST)
print('Compatibility counts:', compatibility.get('counts'))
print('Compatibility:', 'PASS' if compatibility.get('downstream_eligible') is True else 'FAIL')
if candidate_import.get('downstream_eligible') is not True:
    raise RuntimeError('STOP: candidate import 不允许进入下游')
if compatibility.get('downstream_eligible') is not True:
    raise RuntimeError('STOP: compatibility audit 未通过，不进入 Train/Validation')

## Cell 5｜Train Reuse + Train Preparation

先按当前正式数据合同构造 Train provider 与 Stage 6 evaluator，生成 Hybrid Train reuse overlay；随后对完整兼容候选集执行冻结的 Train prefilter。不会重新计算可被精确复用的 Stage 5 Train 指标。

In [ ]:
train_context = build_real_reward_data_context(RealRewardDataConfig(), DATA_PATHS)
provider = RealRewardProvider(train_context, subexpression_cache_max_bytes=0)
provider_manifest = provider.manifest()
provider_fingerprint = provider.fingerprint()
del provider, train_context
gc.collect()

stage6_context = build_stage6_evaluation_context(paths=DATA_PATHS)
evaluator = Stage6CandidateEvaluator(
    stage6_context,
    compatibility_audit_fingerprint=compatibility['audit_fingerprint'],
    accepted_registry_fingerprint=compatibility['accepted_registry_fingerprint'],
)
TRAIN_REUSE_MANIFEST = run_stage6_hybrid_train_reuse_overlay(
    source_set_manifest_path=SOURCE_SET_MANIFEST,
    candidate_import_manifest_path=CANDIDATE_IMPORT_MANIFEST,
    compatibility_manifest_path=COMPATIBILITY_MANIFEST,
    evaluator=evaluator,
    target_provider_manifest=provider_manifest,
    target_provider_fingerprint=provider_fingerprint,
    output_root=STAGE6_ROOT / 'train_reuse',
)
del evaluator, stage6_context, provider_manifest
gc.collect()
print('Train reuse/overlay manifest:', TRAIN_REUSE_MANIFEST)
print('Train reuse counts:', read_json(TRAIN_REUSE_MANIFEST).get('counts'))

train_result = run_current_stage6_train_preparation(
    compatibility_manifest_path=COMPATIBILITY_MANIFEST,
    overlay_manifest_path=TRAIN_REUSE_MANIFEST,
    output_root=STAGE6_ROOT / 'train_preparation',
    progress_callback=compact_progress,
    data_paths=DATA_PATHS,
)
TRAIN_ENTRY_MANIFEST = Path(train_result['entry_manifest_path'])
if train_result['invocation'].get('run_status') != 'complete':
    raise RuntimeError(f"STOP: Train preparation 未完成: {train_result['invocation']}")
if train_result.get('train_pass_manifest_path') is None:
    raise RuntimeError('STOP: Train-pass manifest 未生成')
TRAIN_PASS_MANIFEST = Path(train_result['train_pass_manifest_path'])
train_pass = read_json(TRAIN_PASS_MANIFEST)
print('Train entry manifest:', TRAIN_ENTRY_MANIFEST)
print('Train-pass manifest:', TRAIN_PASS_MANIFEST)
print('Accepted candidates before prefilter:', read_json(TRAIN_ENTRY_MANIFEST).get('candidate_count'))
print('Train prefilter pass:', train_pass.get('train_pass_count'))

## Cell 6｜Fresh Validation

只对 Train-pass candidates 执行当前冻结合同下的 fresh Validation；不读取 Test/OOS。该 Cell 可能耗时较长。

In [ ]:
validation_result = run_current_stage6_validation_evaluation(
    compatibility_manifest_path=COMPATIBILITY_MANIFEST,
    overlay_manifest_path=TRAIN_REUSE_MANIFEST,
    train_entry_manifest_path=TRAIN_ENTRY_MANIFEST,
    train_pass_manifest_path=TRAIN_PASS_MANIFEST,
    output_root=STAGE6_ROOT / 'validation_evaluation',
    progress_callback=compact_progress,
    data_paths=DATA_PATHS,
)
VALIDATION_ENTRY_MANIFEST = Path(validation_result['entry_manifest_path'])
if validation_result['invocation'].get('run_status') != 'complete':
    raise RuntimeError(f"STOP: Validation 未完成: {validation_result['invocation']}")
print('Validation entry manifest:', VALIDATION_ENTRY_MANIFEST)
print('Validation evaluated candidates:', validation_result['candidate_count'])
print('Validation invocation:', validation_result['invocation'])

## Cell 7｜Formal Two-Phase Selection

调用现有正式实现完成六项 hard filter 与 Train directional long-excess Pearson 去相关。最终产物只能称为 **Provisional Factor Pool**。

In [ ]:
SELECTION_MANIFEST = run_current_stage6_two_phase_provisional_selection(
    validation_entry_manifest_path=VALIDATION_ENTRY_MANIFEST,
    compatibility_manifest_path=COMPATIBILITY_MANIFEST,
    overlay_manifest_path=TRAIN_REUSE_MANIFEST,
    output_root=STAGE6_ROOT / 'provisional_selection',
    progress_callback=compact_progress,
    data_paths=DATA_PATHS,
)
selection = read_json(SELECTION_MANIFEST)
if selection.get('engineering_smoke') is not False:
    raise RuntimeError('STOP: selection 不是正式 Provisional selection')
if selection.get('oos') != 'not_loaded_not_evaluated':
    raise RuntimeError('STOP: OOS lock 状态异常')
print('Selection manifest:', SELECTION_MANIFEST)
print('Selection counts:', selection.get('counts'))
print('Result: Provisional Factor Pool（不是 Final/Frozen Baseline Pool）')

## Cell 8｜Execution Summary + Reporting Handoff

下列 7 个路径与 `stage6_reporting.ipynb` 的 loader 参数逐项对应。只有前面所有 Cell 完成后才运行本 Cell。

In [ ]:
candidate_count = candidate_import.get('counts', candidate_import.get('record_counts'))
train_entry = read_json(TRAIN_ENTRY_MANIFEST)
train_pass = read_json(TRAIN_PASS_MANIFEST)
validation_entry = read_json(VALIDATION_ENTRY_MANIFEST)
selection = read_json(SELECTION_MANIFEST)

summary = {
    'Stage 5 source': HYBRID_RUN_ID,
    'Imported candidates': candidate_count,
    'Train preparation candidates': train_entry.get('candidate_count'),
    'Train prefilter pass': train_pass.get('train_pass_count'),
    'Validation evaluated': validation_result.get('candidate_count'),
    'Hard-filter / decorrelation / pool counts': selection.get('counts'),
    'Provisional Factor Pool size': (selection.get('counts') or {}).get('retained'),
}
for key, value in summary.items():
    print(f'{key}: {value}')

STAGE6_REPORTING_INPUTS = {
    'SOURCE_SET_MANIFEST': Path(SOURCE_SET_MANIFEST).resolve(),
    'CANDIDATE_IMPORT_MANIFEST': Path(CANDIDATE_IMPORT_MANIFEST).resolve(),
    'COMPATIBILITY_MANIFEST': Path(COMPATIBILITY_MANIFEST).resolve(),
    'TRAIN_ENTRY_MANIFEST': Path(TRAIN_ENTRY_MANIFEST).resolve(),
    'TRAIN_PASS_MANIFEST': Path(TRAIN_PASS_MANIFEST).resolve(),
    'VALIDATION_ENTRY_MANIFEST': Path(VALIDATION_ENTRY_MANIFEST).resolve(),
    'SELECTION_MANIFEST': Path(SELECTION_MANIFEST).resolve(),
}
print('\n复制到 notebooks/stage6_reporting.ipynb 的 7 个输入路径：')
for name, path in STAGE6_REPORTING_INPUTS.items():
    print(f"{name} = Path(r'{path}')")
print('\nStage 6 正式执行终点：Provisional Factor Pool + complete manifests。')
print('下一步：运行 stage6_reporting.ipynb；人工 C3 review 后再进入 D1。')